## Reference Data

In [1]:
%pip install pytest


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
reference_data = [
  {
    "question": "What’s the leave policy?", 
    "ground_truth": "employees must submit a leave request for approval.", #Expected llm generated answer
    "context": "Employees must submit a leave request for approval. " #Expected retrieved context
  }
]

# reference_data = [
#   {
#     "question": "What is the company's policy on remote work?", 
#     "ground_truth": "Remote work is allowed up to 3 days per week.", #Expected llm generated answer
#     "context": "Remote work is allowed up to 3 days per week." #Expected retrieved context
#   }
# ]
question = reference_data[0]['question']
ground_truth = reference_data[0]['ground_truth']
context = reference_data[0]['context']
print (f"question: {question}")
print (f"ground_truth: {ground_truth}")
print (f"context: {context}")

question: What’s the leave policy?
ground_truth: employees must submit a leave request for approval.
context: Employees must submit a leave request for approval. 


In [3]:
# Retrieve context from Milvus DB

from milvus_chatbot_with_rag import retrieve_similiar_contexts, generate_answer

def perform_retrieval(question):

    retrieved_context = retrieve_similiar_contexts(question, "policy_docs_collection", 1)[0]['content']
    print (f"perform_retrieval.retrieved_context: {retrieved_context}")
    return retrieved_context

# Generate answer using LLM

question = reference_data[0]['question']
context = perform_retrieval(question)
answer = generate_answer(question, context)
answer


/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:28: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


Connected to Milvus on Zilliz Cloud


/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:36: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(collection_name)
I0617 16:50:16.623178 1004267 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7359.31it/s]
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:41: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


perform_retrieval.retrieved_context: Employees must submit a leave request for approval.


'Employees must submit a leave request for approval.'

In [4]:
%pip install ragas datasets 

I0617 16:50:25.038432 1004620 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness

from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

# Question User asked
question = reference_data[0]['question']

# Reference context (should be a string)
reference_context = reference_data[0]['context']

# ground truth answer
ground_truth = reference_data[0]['ground_truth']

# Retrieved context (a string from perform_retrieval)
retrieved_context = [perform_retrieval(question)]
llm_answer = generate_answer(question, retrieved_context[0])

# Build dataset properly
dataset_dict = {
    "question": [question],
    "contexts": [retrieved_context],    # list of strings INSIDE another list
    "ground_truth": [ground_truth],   # single string/ reference answer
    "answer": [llm_answer]
}

print(f"dataset_dict: {dataset_dict}")

ragas_dataset = Dataset.from_dict(dataset_dict)


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_4996/3037382094.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_correctness
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_4996/3037382094.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import faithfulness, answer_correctness
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:28: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


Connected to Milvus on Zilliz Cloud


/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:36: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(collection_name)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6362.21it/s]
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/Session12_ragas_eval/milvus_chatbot_with_rag.py:41: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


perform_retrieval.retrieved_context: Employees must submit a leave request for approval.
dataset_dict: {'question': ['What’s the leave policy?'], 'contexts': [['Employees must submit a leave request for approval.']], 'ground_truth': ['employees must submit a leave request for approval.'], 'answer': ['Employees must submit a leave request for approval.']}


In [6]:
from ragas.llms.base import llm_factory
from ragas import evaluate
from ragas.metrics import answer_correctness

results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_correctness]  
)


print("LLM Generation Evaluation Results:")
results.to_pandas()



/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_4996/84979105.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness
I0617 16:50:38.433135 1005256 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
Evaluating: 100%|██████████| 2/2 [00:10<00:00,  5.04s/it]

LLM Generation Evaluation Results:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness
0,What’s the leave policy?,[Employees must submit a leave request for app...,Employees must submit a leave request for appr...,employees must submit a leave request for appr...,1.0,NaN


In [7]:
from ragas.llms.base import llm_factory
from ragas import evaluate
from ragas.metrics import answer_correctness

# Create the modern LLM wrapper
client = OpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Run evaluation
results = evaluate(
    dataset=ragas_dataset,
    metrics=[answer_correctness],
    llm=llm
)

print("LLM Generation Evaluation Results:")
results.to_pandas()


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_4996/3714285956.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness
Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.61s/it]

LLM Generation Evaluation Results:


,user_input,retrieved_contexts,response,reference,answer_correctness
0,What’s the leave policy?,[Employees must submit a leave request for app...,Employees must submit a leave request for appr...,employees must submit a leave request for appr...,NaN


## 